# Run Broadcasting Experiments

Use this notebook for the main protocol workflow: exact simulation, QEC Monte Carlo sampling, a single IBM hardware point, or an IBM hardware tau sweep. Results are saved through the unified JSON schema in `results/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from broadcasting import (
    ExactBackend,
    HardwareBackend,
    HPCBackend,
    ProtocolConfig,
    SamplingBackend,
    load_run,
    save_run,
)
from broadcasting.analysis import delay_axis
from broadcasting.plotting import (
    autocorrelation_from_run,
    periodogram_from_run,
    plot_fidelity_vs_noise,
    plot_periodicity_comparison,
    plot_run_sweep,
    save_figure,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})


## Configuration

Set `MODE` to `exact`, `sampling`, `hardware`, `hardware_tau_sweep`, or `hpc`. Every mode builds the
same `ProtocolConfig` and hands it to whichever `Backend` subclass matches (`ExactBackend`,
`SamplingBackend`, `HardwareBackend`, `HPCBackend`).


In [ ]:
MODE = "exact"

M = 1
N = 2
alpha = 1.0 / np.sqrt(2)
use_qec = False
outcomes = [0] * M
seed = 42

rng = np.random.default_rng(seed)
nt = 1
theta_samples = rng.uniform(0, 2 * np.pi, size=(nt, M)).tolist()
thetas = theta_samples[0]

p_list = np.linspace(0, 1, 50).tolist()
n_samples = 1000

tau_values = np.linspace(0, 6000, 121).astype(int).tolist()
tau = tau_values[0]

IBM_PROFILE = "mprest1"
IBM_BACKEND = None  # None -> least-busy operational backend; set explicitly for controlled repeats.
OPTIMIZATION_LEVEL = 3
SHOTS = 4096


# Only used when MODE == "hpc": which Backend the cluster job itself should run,
# and whether to actually call `sbatch` (False just builds/prints the command).
HPC_MODE = "exact"
HPC_SUBMIT = False
HPC_ARRAY = False
HPC_CONCURRENCY = None

SAVE_FIGURES = False

FIGURE_DIR = Path("figures")

print(f"Mode={MODE}  M={M}  N={N}  QEC={use_qec}")
print("theta samples:")
for i, sample in enumerate(theta_samples):
    print(f"  {i}: {np.array(sample)}")

## Run


In [ ]:
if MODE == "sampling" and not use_qec:
    raise ValueError("MODE='sampling' requires use_qec=True.")

config = ProtocolConfig(
    M=M,
    N=N,
    alpha=alpha,
    thetas=thetas,
    p_list=p_list if MODE in {"exact", "sampling", "hpc"} else [],
    use_qec=use_qec,
    outcomes_list=outcomes,
    tau=tau if MODE == "hardware" else None,
    n_samples=n_samples if MODE in {"sampling", "hpc"} else None,
    seed=seed,
)

if MODE == "exact":
    backend = ExactBackend()
    result = backend.run(config)
elif MODE == "sampling":
    backend = SamplingBackend(n_samples=n_samples, seed=seed)
    result = backend.run(config)
elif MODE == "hpc":
    backend = HPCBackend(mode=HPC_MODE, array=HPC_ARRAY, concurrency=HPC_CONCURRENCY, submit=HPC_SUBMIT)
    result = backend.run(config)
    print(result.metadata["command"])
elif MODE in {"hardware", "hardware_tau_sweep"}:
    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(name=IBM_PROFILE)
    backend = HardwareBackend(
        service=service,
        backend_name=IBM_BACKEND,
        shots=SHOTS,
        optimization_level=OPTIMIZATION_LEVEL,
    )
    if MODE == "hardware":
        result = backend.run(config)
    else:
        result = backend.run_tau_sweep(config, tau_values, theta_samples=theta_samples)
else:
    raise ValueError(f"Unknown MODE: {MODE}")



print(f"Recorded mode: {result.metadata.get('mode')}")
print(f"Fidelity array shape: {np.asarray(result.fidelities).shape}")

## Save And Plot


In [ ]:
out_path = save_run(result, config)
saved_run = load_run(out_path)
print(f"Saved to {out_path}")

if saved_run["sweep"]["axis"] == "p":
    fids = np.asarray(saved_run["fidelities"], dtype=float)
    fig = plot_fidelity_vs_noise(
        np.asarray(saved_run["sweep"]["values"], dtype=float),
        {f"Receiver {i}": fids[:, i] for i in range(saved_run["N"])},
        mode_label=saved_run.get("backend", MODE),
        protocol_info={"M": M, "N": N, "use_qec": use_qec},
        show=False,
    )
else:
    fig = plot_run_sweep(
        saved_run,
        show=False,
    )

plt.tight_layout()
if SAVE_FIGURES:
    figure_path = FIGURE_DIR / f"{Path(out_path).stem}.png"
    save_figure(fig, figure_path)
    print(f"Saved title-free figure to {figure_path}")
plt.show()


## Optional Exact Vs Sampling Overlay

`SamplingBackend` is for the QEC path, so this comparison runs only when `use_qec=True`.


In [ ]:
RUN_COMPARISON = False

if RUN_COMPARISON and use_qec:
    compare_config = ProtocolConfig(
        M=M,
        N=N,
        alpha=alpha,
        thetas=thetas,
        p_list=p_list,
        use_qec=True,
        outcomes_list=outcomes,
        seed=seed,
    )
    exact = ExactBackend().run(compare_config)
    sampled = SamplingBackend(n_samples=5000, seed=seed).run(compare_config)

    p_arr = np.asarray(p_list, dtype=float)
    exact_avg = np.asarray(exact.fidelities, dtype=float).mean(axis=1)
    sampled_avg = np.asarray(sampled.fidelities, dtype=float).mean(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(p_arr, exact_avg, label="Exact")
    ax.plot(p_arr, sampled_avg, "--", label="Sampling")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.set_xlabel("Depolarizing probability p")
    ax.set_ylabel("Average fidelity")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        save_figure(fig, FIGURE_DIR / "exact_vs_sampling.png")
    plt.show()
elif RUN_COMPARISON:
    print("Set use_qec=True before running the sampling comparison.")


## Optional Sampling Convergence

This is **new simulation data collection**, disabled by default. The historical convergence image was withdrawn because configuration seed 0 overrode every supposedly independent backend seed. The shared helper now sets the effective seed and trajectory count per run, archives all measurements, and labels the log-log fit as descriptive.


In [ ]:
RUN_CONVERGENCE = False

if RUN_CONVERGENCE:
    from scripts.generate_figures import generate_figure_1_convergence

    # Explicit collection: saved independently seeded measurements precede plotting.
    convergence_archive = generate_figure_1_convergence(
        [FIGURE_DIR] if SAVE_FIGURES else [],
        quick=False,
        collect=True,
    )
    print(f"Convergence measurements: {convergence_archive}")


## Optional Saved-Run Periodicity Analysis

Analyze a saved bare-broadcasting delay sweep using autocorrelation and a linearly detrended Hann periodogram. Peaks are exploratory: a single ordered sweep cannot establish a reproducible oscillation or physical cause. This cell reads existing results only and remains disabled by default. Recorded `dt` converts both frequency and spectral-density units; historical runs without `dt` stay in native units.


In [ ]:
RUN_PERIODICITY_ANALYSIS = False

if RUN_PERIODICITY_ANALYSIS:
    target_run = None
    if "saved_run" in locals() and saved_run.get("sweep", {}).get("axis") == "tau" and len(saved_run["sweep"]["values"]) >= 8:
        target_run = saved_run
    else:
        for path in sorted(Path("results").glob("run_*.json"), reverse=True):
            candidate = load_run(path)
            if (candidate.get("experiment_type") == "hardware"
                and candidate.get("sweep", {}).get("axis") == "tau"
                and candidate.get("N") == 2 and not candidate.get("use_qec")
                and len(candidate["sweep"]["values"]) >= 8):
                target_run = candidate
                print(f"Loaded saved tau sweep: {path}")
                break

    if target_run is None:
        print("No saved hardware delay sweep with at least eight points was found.")
    else:
        from broadcasting.analysis import periodicity_summary

        scale, unit = delay_axis(target_run)
        fig = plot_periodicity_comparison(
            [target_run], [target_run.get("filename", "saved run")],
            tau_scale=scale, tau_label=f"Idle delay ({unit})", show=False,
        )
        plt.tight_layout()
        if SAVE_FIGURES:
            save_figure(fig, FIGURE_DIR / "delay_periodicity_analysis.png")
        plt.show()
        print(periodicity_summary(target_run))
